# WebQSP with Neo4j-backed graph input

This example keeps the complete WebQSP graph in an already-populated Neo4j database. FastRLM gives the graph child a bounded read-only Cypher operation; every query result is returned to it as a NetworkX `MultiDiGraph`. Set `NEO4J_PASSWORD` and `RLM_MODEL` in the environment before running, and use a Neo4j account with server-enforced read-only privileges.

Entity mentions are resolved through the pre-created `webqsp_entity_names` full-text index. For example, the child turns `Ben Franklin` into the parameterized Lucene query `ben* AND franklin*`, then reranks candidates using matching relationship predicates before retrieving answer evidence.

In [1]:
import os
from typing import Literal
import json

import fast_rlm
from fast_rlm import Neo4jGraph, RLMConfig
from pydantic import BaseModel

In [2]:
# Constants: change the question/example pair together.
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j-admin-2026")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
FULLTEXT_INDEX = "webqsp_entity_names"
RLM_MODEL = os.getenv("RLM_MODEL", "poolside/laguna-s-2.1:free")
# os.environ.setdefault("RLM_MODEL_BASE_URL", "http://localhost:11434/v1")
# os.environ.setdefault("RLM_MODEL_API_KEY", "ollama")
QUESTION = "what does jamaican people speak"
Q_ENTITY = ["Jamaica"]
GOLD_ANSWERS = ["Jamaican English", "Jamaican Creole English Language"]
WEBQSP_SCHEMA = """
(:WebQSPExample {id, source_id, question, split, q_entity_json})
(:WebQSPEntity {key, name})
(:WebQSPEntity)-[:RELATED {predicate}]->(:WebQSPEntity)
""".strip() + f"\nFULLTEXT INDEX {FULLTEXT_INDEX} FOR (:WebQSPEntity) ON (name)"

In [3]:
print(RLM_MODEL)

deepseek/deepseek-v4.1-flash


In [4]:
class WebQSPAnswer(BaseModel):
    answer_entities: list[str]
    status: Literal["answered", "insufficient_data"]


graph = Neo4jGraph(
    uri=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    schema=WEBQSP_SCHEMA,
)
config = RLMConfig(
    primary_agent=RLM_MODEL,
    sub_agent=RLM_MODEL,
    max_depth=3,
    max_graph_queries=1,
    max_graph_hops=4,
    max_cypher_queries=10,
    max_cypher_rows=100,
    cypher_timeout_ms=15_000,
)

In [5]:
result = fast_rlm.run(
    {
        "question": QUESTION,
    },
    graph=graph,
    config=config,
    output_schema=WebQSPAnswer,
    evaluation_q_entity=Q_ENTITY,
    instruction=(
        f"Delegate graph retrieval once. Resolve complete question entity mentions through "
        f"the {FULLTEXT_INDEX} full-text index using a parameterized Lucene prefix query. "
        "Rerank candidates with their incident relationship predicates, then scope answer "
        "queries to selected stable key values. Return only answer-supporting evidence."
    ),
    # verbosity="summary",
)

answer = result["results"]
with open(result["log_file"]) as log_handle:
    events = [json.loads(line) for line in log_handle if line.strip()]
audit_events = [
    event for event in events
    if event.get("event_type", "").startswith(("graph_", "neo4j_"))
]
model_calls = sum(
    event.get("usage", {}).get("total_tokens", 0) > 0
    for event in events
)
entity_evaluation = next(
    (event for event in reversed(audit_events) if "entity_precision" in event),
    {},
)

✔ Neo4j connected: database=neo4j
📝 Logging to: /Users/kapish/Kapish/RP/fast-rlm/examples/logs/run_2026-09-24T22-21-08-079Z.jsonl

✔ Python Ready
✔ requests + httpx ready (env setup took 1330ms)


⠋ Generating code...
✔ Code generated


↳ graph_query called
✔ Python Ready
✔ requests + httpx ready (env setup took 1068ms)


⠋ Generating code...
✔ Code generated


    │ ────────────────────────── Depth 1 · Step 1/20 ──────────────────────────
    │ ╭ Reasoning ─────────────────────────────────────────────────────────────╮
    │ │ We need to answer "what does jamaican people speak". Let's do a        │
    │ │ full-text entity search for "Jamaican" / "Jamaica" entities.           │
    │ │                                                                        │
    │ │ Use build_fulltext_entity_query or find_neo4j_entity_seeds. Let's      │
    │ │ start with a fulltext query for "Jamaican people" or "Jamaica".        │
    │ │                                                                        │
    │ │ Let me use build_fulltext_entity_queries with entity_mentions.         │
    │ │ Actually the simplest: use build_fulltext_entity_query("Jamaica").     │
    │ │                                                                        │
    │ │ But let's think: the question entity is "Jamaican people" — likely the │
    │ │ WebQSP entity "Jamaic

...

✔ Code generated
⠋ Generating code...


    │ ────────────────────────── Depth 1 · Step 8/20 ──────────────────────────
    │ ╭ Reasoning ─────────────────────────────────────────────────────────────╮
    │ │ The GRAPH_FINAL failed. Maybe I need to include nodes/edges with       │
    │ │ proper keys. The issue may be that nodes need "key" properties, which  │
    │ │ I set. Let me check - perhaps the graph needs to be a MultiDiGraph but │
    │ │ the error is generic. Let me try rebuilding using copies from          │
    │ │ evidence_graph... but those nodes weren't added to evidence_graph      │
    │ │ (only the observation of Jamaica relationship was rows with no nodes). │
    │ │                                                                        │
    │ │ Actually the error might be because GRAPH_FINAL expects specific node  │
    │ │ key format. Let me reconsider. The error "Neo4j graph REPL execution   │
    │ │ failed" - maybe it requires that the answer graph nodes come from      │
    │ │ evidence_graph, or th

✔ Code generated
⠋ Generating code...


───────────────────────────── Depth 0 · Step 2/20 ─────────────────────────────
╭ Reasoning ───────────────────────────────────────────────────────────────────╮
│ The projection isn't showing node properties. Let me inspect the subgraph    │
│ directly.                                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯
╭ Python Code ─────────────────��───────────────────────────────────────────────╮
│ 1 import networkx as nx                                                      │
│ 2 print("nodes:", list(subgraph.nodes(data=True)))                           │
│ 3 print("edges:", list(subgraph.edges(data=True)))                           │
╰──────────────────────────────────────────────────────────────────────────────╯
╭ Result ──────────────────────────────────────────────────────────────────────╮
│ [FULL OUTPUT SHOWN]... nodes: [3]:                                           │
│   [2]:                    

✔ Code generated


In [ ]:
print("Question:", QUESTION)
print("Prediction:", answer["answer_entities"])
print("Status:", answer["status"])
print("Golden answer:", GOLD_ANSWERS)
print("Entity precision:", entity_evaluation.get("entity_precision"))
print("Entity recall:", entity_evaluation.get("entity_recall"))
print("Seed entities considered:", entity_evaluation.get("seed_entities_considered", []))
print("Selected seed entities:", entity_evaluation.get("selected_seed_entities", []))
print("Model calls:", model_calls)
print("Total tokens:", result["usage"]["total_tokens"])
print("Graph-native audit:")
for event in audit_events:
    fields = []
    details = event.get("graph", {})
    if details:
        fields.append(
            f"graph={details.get('backend')}/{details.get('graph_type', 'source')} "
            f"nodes={details.get('nodes', '-')}, edges={details.get('edges', '-')}"
        )
    for key in (
        "cypher_query_number", "parameter_names", "row_limit", "duration_ms",
        "rows", "truncated", "max_graph_hops", "entity_mentions",
        "candidate_count", "candidates", "min_entity_confidence",
        "entity_precision", "entity_recall", "seed_source", "seed_count",
        "seed_entities_considered", "selected_seed_entities",
        "seeds", "seed_match_methods", "error", "error_code",
    ):
        if key in event:
            fields.append(f"{key}={event[key]}")
    print(f"  {event['event_type']}: " + ", ".join(fields))